# ANIMA — Microtonal Chord Generation (V3, modelA_hybrid_v1)

Generate 53-TET microtonal chord progressions with the trained hybrid L1+L2
GPT (`checkpoints/modelA_hybrid_v1/best.pt`). EigenSpace `(α, β, γ, D)`
is injected as **positional embedding** (MLP 4 → 768, added to token
embedding), exactly as during training.

Pipeline
1. Load model + V3 vocabulary + EigenSpace prior (sampled from packed training eigen at `CHORD_START` positions)
2. Build prompt from `TYPE_* / STYLE_* / TONALITY_* / FORM_*` (all V3 tokens)
3. Autoregressive sampling; per-token eigen is kept in sync with the stream:
   - `<start>`, header, pad → `HEADER_EIGEN = [1, 1, 2, 0]`
   - L1 chord block (between `.` and `CHORD_START`) → sample from prior
   - Inside `CHORD_START…CHORD_END` → inherit, then after `CHORD_END` recompute from actual PV_ pitches and back-fill the whole chord span
   - Structural bridges (`|`, `|:`, `:|`, `FORM_*`) → inherit previous chord's eigen (Visibility Rule B)
4. Export to MPE MIDI, visualize, render audio


In [ ]:
import sys, os, time, importlib
import numpy as np
import torch
from pathlib import Path
from IPython.display import Audio, display

SRC_DIR = Path('.').resolve()
ROOT_DIR = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))

import play_mpe as pm
import midi_viz as mv

print(f'Root: {ROOT_DIR}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 1. Load Model

In [ ]:
import generate as gen
importlib.reload(gen)

# Load V3 vocabulary (vocab_size = 2930, block_size = 4096)
VOCAB_PATH = ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'
vocab = gen.Vocabulary(str(VOCAB_PATH))
print(f'Vocabulary: {len(vocab)} tokens')

# Load the trained modelA_hybrid_v1 (Large / GPT-2-small class)
CHECKPOINT_PATH = ROOT_DIR / 'checkpoints' / 'modelA_hybrid_v1' / 'best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt = gen.load_checkpoint(str(CHECKPOINT_PATH), device, vocab.vocab_size)

# EigenSpace computer (legacy, uses min(pitches) as root — fine for inference
# recompute after CHORD_END)
from eigenspace import EigenSpaceComputer
eigen_computer = EigenSpaceComputer()

# Empirical EigenSpace prior — samples real (α, β, γ, D) vectors from
# packed training data at CHORD_START positions
TOKENIZED_DIR = ROOT_DIR / 'dataset' / 'tokenized'
eigen_prior = gen.EigenSpacePrior(str(TOKENIZED_DIR), max_seqs=2000)

# Tokenizer (for proper MPE MIDI export with RPN pitch-bend setup)
from tokenizer import MPETokenizer
tokenizer = MPETokenizer()

print(f'\nDevice: {device}')
print(f'EigenSpace prior: {len(eigen_prior)} chord vectors (from training)')
print(f'Checkpoint iter: {ckpt.get("iter_num")}, best_val_loss: {ckpt.get("best_val_loss"):.4f}')
print('Model loaded ✓')


## 2. Generation Settings

V3 vocabulary (see `dataset/tokenized/vocab.json`):

- `TYPE_*` (14): `0_major 0_minor 1_minor 1_neutral 2_minor 2_subminor 3_major 3_minor 4_minor 4_upmajor 5_major_v2 5_minor 6_minor 6_neutral_n`
- `STYLE_*` (16, capitalized): `Jazz Blues Folk Bossa Reggae Samba Funk Pop Son Rock Soul Balad RnB Gospel Afoxé "Even 8ths"`
- `TONALITY_*` (34): e.g. `C_major`, `A_minor`, …
- `FORM_*` (9): `INTRO A B C D VERSE HEAD CODA SEGNO`
- Structural: `. | |: :| /`
- L1 chord block: `. DUR_<d> R_<root> Q_<quality> [X_<ext> …] [/ R_<bass>]`
- L2 chord block: `CHORD_START DUR_<d> PV_<step>_<vel> … CHORD_END`


In [ ]:
# ── Generation parameters ──
MAX_TOKENS    = 1024
TEMPERATURE   = 0.9
TOP_K         = 40
TOP_P         = 0.95
SEED          = None

# ── Conditioning (V3 token prefixes) ──
# Set any to None to skip that condition.
TYPE_LABEL     = '5_major_v2'   # -> TYPE_5_major_v2
STYLE_LABEL    = 'Blues'        # -> STYLE_Blues   (NOTE: capitalized)
TONALITY_LABEL = 'C_major'      # -> TONALITY_C_major
FORM_LABEL     = 'A'            # -> FORM_A
START_REPEAT   = True           # emit '|:' after FORM_* so the model opens a section

# Optional symbolic L1 seed — a single chord block written in V3 tokens.
# Leave as None for free generation from header-only prompt.
# Example: '. DUR_4.0 R_C Q_maj7'
FIRST_CHORD_L1 = None

# ── Audio rendering ──
PLAYBACK_SPEED = 1.2
WAVEFORM       = 'square'   # 'sine', 'square', 'sawtooth', 'triangle'
REVERB         = 33
MIDI_TEMPO     = 160
SAVE_AUDIO     = True

# ── Output ──
OUTPUT_DIR  = ROOT_DIR / 'dataset' / 'generated'
MIDI_DIR    = OUTPUT_DIR / 'midi'
AUDIO_DIR   = OUTPUT_DIR / 'audio'
MIDI_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print('Settings configured ✓')
print(f'  Max tokens:  {MAX_TOKENS}')
print(f'  Temperature: {TEMPERATURE}  top_k={TOP_K}  top_p={TOP_P}')
print(f'  TYPE:        {TYPE_LABEL}')
print(f'  STYLE:       {STYLE_LABEL}')
print(f'  TONALITY:    {TONALITY_LABEL}')
print(f'  FORM:        {FORM_LABEL}  (open repeat: {START_REPEAT})')
print(f'  L1 seed:     {FIRST_CHORD_L1 or "(none)"}')
print(f'  MIDI dir:    {MIDI_DIR}')
print(f'  Audio dir:   {AUDIO_DIR}')


## 3. Generate

EigenSpace is the model's positional embedding (`EigenSpacePositionalEncoding`
in `src/model.py`, MLP 4 → 768). At inference we must therefore feed a
`(T, 4)` array alongside the token stream. Below we do this consistently
with how training data was packed (Visibility Rule B): per-token `(α,β,γ,D)`
is held constant inside a chord span, inherited across structurals, and
`HEADER_EIGEN = [1, 1, 2, 0]` elsewhere.


In [ ]:
import torch.nn.functional as F

# HEADER_EIGEN must match pack_data_v3.py (Rule B).
HEADER_EIGEN = np.array([1.0, 1.0, 2.0, 0.0], dtype=np.float32)


def _build_prompt_tokens():
    toks = ['<start>']
    # Song-level header order matches sidecars: <style> STYLE_* <tonality> TONALITY_* TYPE_*
    if STYLE_LABEL is not None:
        tok = f'STYLE_{STYLE_LABEL}'
        assert tok in vocab.token_to_id, f'Unknown style token: {tok}'
        toks += ['<style>', tok]
    if TONALITY_LABEL is not None:
        tok = f'TONALITY_{TONALITY_LABEL}'
        assert tok in vocab.token_to_id, f'Unknown tonality token: {tok}'
        toks += ['<tonality>', tok]
    if TYPE_LABEL is not None:
        tok = f'TYPE_{TYPE_LABEL}'
        assert tok in vocab.token_to_id, f'Unknown type token: {tok}'
        toks.append(tok)
    if FORM_LABEL is not None:
        tok = f'FORM_{FORM_LABEL}'
        assert tok in vocab.token_to_id, f'Unknown form token: {tok}'
        toks.append(tok)
    if START_REPEAT:
        toks.append('|:')
    if FIRST_CHORD_L1 is not None:
        for t in FIRST_CHORD_L1.split():
            assert t in vocab.token_to_id, f'Unknown token in FIRST_CHORD_L1: {t}'
            toks.append(t)
    return toks


@torch.no_grad()
def generate_v3(model, vocab, eigen_computer, prompt_tokens,
                max_new_tokens, temperature, top_k, top_p, device,
                eigen_prior):
    """
    V3-aware autoregressive generation.

    Maintains a parallel eigen array (T, 4) so that at each step the model
    sees per-token (alpha, beta, gamma, D) consistent with how training
    data was packed.

    State machine per token (matches pack_data_v3.py Visibility Rule B):
      - 'header' (<start>, <style>, STYLE_*, <tonality>, TONALITY_*, TYPE_*,
        FORM_* before any chord, padding)                     -> HEADER_EIGEN
      - 'in_l1_block'  (seen '.', not yet CHORD_END)          -> sampled-from-prior
      - 'in_l2_block'  (CHORD_START..CHORD_END)               -> sampled, then
        retroactively overwritten with true (alpha,beta,gamma,D) once
        CHORD_END is emitted (EigenSpaceComputer.compute_for_tokens)
      - 'bridge'  (|, |:, :|, /, FORM_* between chords)       -> inherit last chord
    """
    model.eval()
    block_size = model.config.block_size

    ids = vocab.encode(prompt_tokens)
    toks = list(prompt_tokens)

    # Initial per-token eigen: HEADER for every prompt token (prompt is
    # pure header/structural at this stage).
    eig = np.tile(HEADER_EIGEN, (len(toks), 1)).astype(np.float32)

    # Track the last chord's eigen for bridges; start with HEADER.
    last_chord_eigen = HEADER_EIGEN.copy()
    # Track the start index of the current L1 chord span (from '.')
    current_span_start = None   # int or None
    current_span_eigen = None   # (4,) sampled-from-prior while span open

    STRUCT_BRIDGE = {'|', '|:', ':|', '/'}

    for step in range(max_new_tokens):
        # Crop to block_size
        T = len(ids)
        lo = max(0, T - block_size)
        idx_t = torch.tensor([ids[lo:]], dtype=torch.long, device=device)
        eig_t = torch.tensor(eig[lo:][None, ...], dtype=torch.float32, device=device)

        logits, _ = model(idx_t, eigen=eig_t)
        logits = logits[:, -1, :] / max(1e-6, temperature)

        if top_k is not None and top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        if top_p is not None and 0 < top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            mask = cum - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[mask] = -float('Inf')
            logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = F.softmax(logits, dim=-1)
        next_id = int(torch.multinomial(probs, num_samples=1).item())
        next_tok = vocab.id_to_token.get(next_id, '<pad>')

        ids.append(next_id)
        toks.append(next_tok)

        # ── Decide the eigen for this newly appended token ──
        if next_tok == '.':
            # Start of a new L1 chord block → sample a candidate eigen from
            # the training prior. It will be retroactively replaced by the
            # true value once the L2 block closes with CHORD_END.
            current_span_eigen = eigen_prior.sample().astype(np.float32)
            current_span_start = len(toks) - 1
            row = current_span_eigen
        elif next_tok == 'CHORD_END':
            # L2 block just closed. Recompute true eigen from actual PV_ pitches
            # and back-fill the entire L1+L2 span (from the leading '.' onwards).
            row = current_span_eigen if current_span_eigen is not None else last_chord_eigen
            eig = np.concatenate([eig, row[None, :]], axis=0)
            # Retroactive recompute — EigenSpaceComputer fills CHORD_START..CHORD_END
            recomputed = eigen_computer.compute_for_tokens(toks)
            # Take the (alpha,beta,gamma,D) of the just-closed chord
            cs_i = None
            for j in range(len(toks) - 1, -1, -1):
                if toks[j] == 'CHORD_START':
                    cs_i = j
                    break
            if cs_i is not None:
                true_eig = recomputed[cs_i].astype(np.float32)
                span_lo = current_span_start if current_span_start is not None else cs_i
                eig[span_lo:len(toks)] = true_eig
                last_chord_eigen = true_eig
            current_span_start = None
            current_span_eigen = None
            if next_id == vocab.end_id:
                break
            continue  # eig already appended
        elif current_span_eigen is not None:
            # Inside an open L1+L2 chord span (between '.' and CHORD_END)
            row = current_span_eigen
        elif next_tok in STRUCT_BRIDGE or next_tok.startswith('FORM_'):
            # Bridge between chords → inherit previous chord's eigen (Rule B)
            row = last_chord_eigen
        else:
            # Header-like / unknown structural → HEADER_EIGEN
            row = HEADER_EIGEN

        eig = np.concatenate([eig, row[None, :]], axis=0)

        if next_id == vocab.end_id:
            break

    return ids, toks, eig


# ── Run ──
if SEED is not None:
    torch.manual_seed(SEED); np.random.seed(SEED)
    print(f'Seed: {SEED}')

prompt_tokens = _build_prompt_tokens()
print(f'Prompt ({len(prompt_tokens)} tok): {" ".join(prompt_tokens)}')
print(f'Generating up to {MAX_TOKENS} new tokens...')

t0 = time.time()
gen_ids, gen_tokens, gen_eigen = generate_v3(
    model, vocab, eigen_computer, prompt_tokens,
    max_new_tokens=MAX_TOKENS,
    temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
    device=device, eigen_prior=eigen_prior,
)
dt = time.time() - t0
new_count = len(gen_ids) - len(prompt_tokens)

n_chords = sum(1 for t in gen_tokens if t == 'CHORD_START')
n_bars   = sum(1 for t in gen_tokens if t == 'BAR')
print(f'\nGenerated {new_count} tokens in {dt:.2f}s ({new_count/max(1e-6,dt):.0f} tok/s)')
print(f'  CHORD_START count: {n_chords}')
print(f'  BAR count:         {n_bars}')
print(f'  Eigen array shape: {gen_eigen.shape}')


## 4. Chord Quality Analysis
Verify the model produces real harmonic chords — not random notes.

In [ ]:
from collections import Counter


def extract_v3_chords(tokens):
    """Walk a V3 stream and return a list of dicts per L1+L2 chord pair.

    Each dict contains:
      l1_root, l1_quality, l1_extensions, l1_bass (from L1 block: '. DUR R Q [X..] [/ R]')
      duration, pitches (list[int]), velocities (list[int])  (from L2 block)
    """
    chords = []
    i = 0
    N = len(tokens)
    while i < N:
        t = tokens[i]
        if t == '.':
            c = dict(l1_root=None, l1_quality=None, l1_extensions=[], l1_bass=None,
                     duration=None, pitches=[], velocities=[])
            i += 1
            # L1 block until CHORD_START
            while i < N and tokens[i] != 'CHORD_START':
                tt = tokens[i]
                if tt.startswith('DUR_'):
                    c['duration'] = float(tt[4:])
                elif tt.startswith('R_') and c['l1_root'] is None:
                    c['l1_root'] = tt[2:]
                elif tt.startswith('Q_'):
                    c['l1_quality'] = tt[2:]
                elif tt.startswith('X_'):
                    c['l1_extensions'].append(tt[2:])
                elif tt == '/':
                    # next R_ is the bass
                    if i + 1 < N and tokens[i+1].startswith('R_'):
                        c['l1_bass'] = tokens[i+1][2:]
                        i += 1
                i += 1
            # L2 block
            if i < N and tokens[i] == 'CHORD_START':
                i += 1
                while i < N and tokens[i] != 'CHORD_END':
                    tt = tokens[i]
                    if tt.startswith('PV_'):
                        parts = tt.split('_')
                        c['pitches'].append(int(parts[1]))
                        c['velocities'].append(int(parts[2]))
                    elif tt.startswith('DUR_') and c['duration'] is None:
                        c['duration'] = float(tt[4:])
                    i += 1
                if i < N:
                    i += 1  # skip CHORD_END
            chords.append(c)
        else:
            i += 1
    return chords


chords = extract_v3_chords(gen_tokens)
n_bars = sum(1 for t in gen_tokens if t == 'BAR')
print(f'--- Stream summary ---')
print(f'  L1+L2 chord pairs: {len(chords)}')
print(f'  BAR markers:       {n_bars}')

# Notes-per-chord
pitch_counts = Counter(len(c['pitches']) for c in chords)
print('\n--- Notes per chord ---')
for k in sorted(pitch_counts):
    pct = 100 * pitch_counts[k] / max(1, len(chords))
    print(f'  {k} notes: {pitch_counts[k]:3d} ({pct:5.1f}%) {"#"*int(pct/2)}')
avg = sum(k*v for k,v in pitch_counts.items()) / max(1, len(chords))
print(f'  Avg: {avg:.2f} notes/chord')

# Duration distribution
dur_counts = Counter(c['duration'] for c in chords if c['duration'] is not None)
print('\n--- Duration distribution ---')
for d in sorted(dur_counts):
    pct = 100 * dur_counts[d] / max(1, len(chords))
    print(f'  DUR_{d:<5}: {dur_counts[d]:3d} ({pct:5.1f}%) {"#"*int(pct/2)}')

# L1 quality histogram (top 10)
qual_counts = Counter(c['l1_quality'] for c in chords if c['l1_quality'])
print('\n--- Top L1 qualities ---')
for q, cnt in qual_counts.most_common(10):
    print(f'  Q_{q:<15}: {cnt}')

# L1 root histogram (top 10)
root_counts = Counter(c['l1_root'] for c in chords if c['l1_root'])
print('\n--- Top L1 roots ---')
for r, cnt in root_counts.most_common(10):
    print(f'  R_{r:<6}: {cnt}')

# First 10 chords readable
print('\n--- First 10 chords (L1 symbol + L2 voicing) ---')
for idx, c in enumerate(chords[:10]):
    ext = (' ' + ' '.join(c['l1_extensions'])) if c['l1_extensions'] else ''
    slash = f"/{c['l1_bass']}" if c['l1_bass'] else ''
    print(f'  {idx+1:2d}. dur={c["duration"]} '
          f'R_{c["l1_root"] or "?"} Q_{c["l1_quality"] or "?"}{ext}{slash}  '
          f'pitches={c["pitches"]}')

# EigenSpace verification — sanity check that eigen is non-trivial and
# CHORD_START positions have been updated away from the HEADER default
cs_positions = [i for i, t in enumerate(gen_tokens) if t == 'CHORD_START']
if cs_positions:
    cs_eigens = gen_eigen[cs_positions]
    uniq = len({tuple(e) for e in cs_eigens.tolist()})
    print(f'\n--- EigenSpace at CHORD_START ({len(cs_positions)} positions) ---')
    print(f'  Unique (alpha,beta,gamma,D) vectors: {uniq}')
    print(f'  alpha range: [{cs_eigens[:,0].min():.3f}, {cs_eigens[:,0].max():.3f}]')
    print(f'  beta  range: [{cs_eigens[:,1].min():.3f}, {cs_eigens[:,1].max():.3f}]')
    print(f'  gamma range: [{cs_eigens[:,2].min():.3f}, {cs_eigens[:,2].max():.3f}]')
    print(f'  D     range: [{cs_eigens[:,3].min():.3f}, {cs_eigens[:,3].max():.3f}]')


## 5. Export to MPE MIDI
Uses the tokenizer's proper MPE export with RPN pitch bend range setup.

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
midi_filename = f'generated_{timestamp}.mid'
midi_path = MIDI_DIR / midi_filename

# Decode tokens back to chord events, then write MPE MIDI with proper RPN
decoded_chords = tokenizer.decode(gen_tokens)

if decoded_chords:
    tokenizer.chords_to_midi(decoded_chords, str(midi_path), tpb=960, tempo_bpm=MIDI_TEMPO)
    n_notes = sum(len(c['notes']) for c in decoded_chords)
    print(f'Exported MPE MIDI: {midi_path.name}')
    print(f'  {len(decoded_chords)} chords, {n_notes} notes, {n_bars} bars')
    print(f'  Tempo: {MIDI_TEMPO} BPM, RPN pitch bend range: +/-2 semitones')
else:
    print('No chords decoded from the generated sequence.')

## 6. Visualize

In [ ]:
if midi_path.exists():
    fig = mv.visualize_midi(str(midi_path), speed=PLAYBACK_SPEED, max_duration=120)
    if fig:
        fig.update_layout(title=f'Generated: {midi_filename}', height=400)
        fig.show()

## 7. Play Audio
Render MIDI to audio for listening.

In [ ]:
if midi_path.exists():
    importlib.reload(pm)
    wav_path = AUDIO_DIR / midi_path.with_suffix('.wav').name if SAVE_AUDIO else None
    audio_data, sr = pm.render_mpe_to_audio_data(
        str(midi_path), speed=PLAYBACK_SPEED,
        waveform=WAVEFORM, reverb=REVERB,
        save_path=wav_path,
    )
    if audio_data is not None:
        display(Audio(audio_data, rate=sr))

## 8. Batch Generate
Generate multiple samples with the same conditioning.

In [ ]:
NUM_SAMPLES = 3

for i in range(NUM_SAMPLES):
    print(f'\n{"="*60}\nSample {i+1}/{NUM_SAMPLES}\n{"="*60}')

    prompt = _build_prompt_tokens()
    ids, toks, eig = generate_v3(
        model, vocab, eigen_computer, prompt,
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
        device=device, eigen_prior=eigen_prior,
    )

    sample_chords = extract_v3_chords(toks)
    sample_bars   = sum(1 for t in toks if t == 'BAR')
    print(f'  {len(sample_chords)} chords, {sample_bars} bars, {len(ids)} tokens')

    sample_path = MIDI_DIR / f'generated_{timestamp}_sample{i+1}.mid'
    decoded = tokenizer.decode(toks)
    if decoded:
        tokenizer.chords_to_midi(decoded, str(sample_path), tpb=960, tempo_bpm=MIDI_TEMPO)
        print(f'  Exported: {sample_path.name}')

    if sample_path.exists():
        wav_path = AUDIO_DIR / sample_path.with_suffix('.wav').name if SAVE_AUDIO else None
        audio_data, sr = pm.render_mpe_to_audio_data(
            str(sample_path), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
            save_path=wav_path,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))

print(f'\n{"="*60}\nAll {NUM_SAMPLES} samples generated.')


## 9. Compare with Training Data

In [ ]:
import random

DATASET_MIDI_DIR = ROOT_DIR / 'dataset' / 'midi_files' / '53_tet_mpe'

if DATASET_MIDI_DIR.exists():
    if TYPE_LABEL is not None:
        type_dir = DATASET_MIDI_DIR / f'type_{TYPE_LABEL}'
    else:
        type_dirs = sorted(DATASET_MIDI_DIR.glob('type_*'))
        type_dir = random.choice(type_dirs) if type_dirs else DATASET_MIDI_DIR

    midi_files = sorted(type_dir.glob('*.mid')) if type_dir.exists() else []
    if midi_files:
        ref_file = random.choice(midi_files)
        print(f'Reference file: {ref_file.relative_to(ROOT_DIR)}')

        fig = mv.visualize_midi(str(ref_file), speed=PLAYBACK_SPEED, max_duration=60)
        if fig:
            fig.update_layout(
                title=f'Training data — {ref_file.parent.name}/{ref_file.stem}',
                height=400,
            )
            fig.show()

        audio_data, sr = pm.render_mpe_to_audio_data(
            str(ref_file), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))
    else:
        print(f'No MIDI files under {type_dir}')
else:
    print(f'Dataset MIDI path not found: {DATASET_MIDI_DIR}')


## 10. Listening-Test Batch (8 clips)

Generates the **Model-A third** of the 24-clip listening test — 8 samples on
the same `(style, transformation)` grid as notebook 13 (2 samples per style
across `{jazz, blues, bossa, rock}` × 8 distinct transformation types).

All clips share the dataset render settings so the three sources are A/B/dataset-
comparable: **17 bars / 170 BPM / Rhodes / reverb=33** via FluidSynth.

Outputs:
- `dataset/audio/listening_test/model_A/*.wav`
- `dataset/listening_test/model_A/midi/*.mid`
- `dataset/listening_test/model_A/manifest.json`

Requires cells 1–3 (model + prior loaded).


In [ ]:
import json
from pathlib import Path

# ── Listening-test grid — must mirror notebook 13 (dataset source) ──
LT_SELECTION = [
    # (style_tag, STYLE_* label, TYPE_* transformation)
    ("jazz",  "Jazz",  "5_major_v2"),
    ("jazz",  "Jazz",  "5_minor"),
    ("blues", "Blues", "2_subminor"),
    ("blues", "Blues", "6_neutral_n"),
    ("bossa", "Bossa", "1_neutral"),
    ("bossa", "Bossa", "3_major"),
    ("rock",  "Rock",  "4_upmajor"),
    ("rock",  "Rock",  "4_minor"),
]
assert len(LT_SELECTION) == 8
assert {s[0] for s in LT_SELECTION} == {"jazz", "blues", "bossa", "rock"}
assert len({s[2] for s in LT_SELECTION}) == 8

# ── First-chord seed quality, keyed by transformation type ──
# Matches the 53-EDO chord-quality convention in `src/chord_mapping.py`:
#   12-TET major                      -> maj7
#   neutral   (root+5+neutral+N7)     -> NN7
#   subminor  (root+5+subminor+sm7)   -> smsm7
#   downmajor (root+5+downmajor+vM7)  -> vMvM7
#   downminor (root+5+downminor+vm7)  -> vmvm7
#   upmajor   (root+5+upmajor+^7)     -> ^^7
# Every seed becomes `. DUR_<dur> R_<root> Q_<quality>` so the model opens
# the progression on a chord consistent with its TYPE_* conditioning.
LT_SEED_QUALITY_BY_TYPE = {
    "1_neutral":   "NmNm7",   # neutral-minor triad + N7   (type_1_neutral)
    "2_subminor":  "smsm7",   # subminor triad + sm7       (type_2_subminor)
    "3_major":     "maj7",    # major triad + M7           (type_3_major)
    "4_minor":     "m7",      # minor triad + m7           (type_4_minor)
    "4_upmajor":   "^^7",     # upmajor triad + ^7         (type_4_upmajor)
    "5_major_v2":  "vMvM7",   # downmajor triad + vM7      (type_5_major_v2)
    "5_minor":     "vmvm7",   # downminor triad + vm7      (type_5_minor)
    "6_neutral_n": "NN7",     # neutral-major triad + N7   (type_6_neutral_n)
}
assert set(LT_SEED_QUALITY_BY_TYPE) == {s[2] for s in LT_SELECTION}

# ── Listening-test render + trim settings (match notebook 13 exactly) ──
LT_SOURCE        = "model_A"
LT_N_BARS        = 17
LT_BEATS_PER_BAR = 4
LT_MAX_BEATS     = LT_N_BARS * LT_BEATS_PER_BAR      # 68 beats
LT_TEMPO_BPM     = 190
LT_RENDER_SPEED  = 1.2
LT_WAVEFORM      = "rhodes"
LT_REVERB        = 33
LT_TONALITY      = "C_major"
LT_FORM          = "A"
LT_MAX_TOKENS    = 2048      # plenty for 17 bars of chords
LT_SEED_ROOT     = "E"       # root of the first-chord seed (per listening-test spec; TONALITY_ remains C_major)
LT_SEED_DUR      = "4.0"     # whole-bar seed

# ── Output layout: per-source listening-test subfolders ──
LT_AUDIO_DIR = ROOT_DIR / "dataset" / "audio" / "listening_test" / LT_SOURCE
LT_OUT_DIR   = ROOT_DIR / "dataset" / "listening_test" / LT_SOURCE
LT_MIDI_DIR  = LT_OUT_DIR / "midi"
LT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
LT_MIDI_DIR.mkdir(parents=True, exist_ok=True)


def _lt_build_prompt(style_label: str, type_label: str) -> list[str]:
    """Header prompt with STYLE_*, TONALITY_*, TYPE_*, FORM_*, |: plus a
    type-appropriate first-chord seed `. DUR_<dur> R_<root> Q_<quality>`.

    The seed ensures every clip opens on a chord whose quality is
    consistent with the requested TYPE_* (e.g. type_3_major starts with
    `Q_vMvM7`, type_2_subminor with `Q_smsm7`, type_4_upmajor with `Q_^^7`).
    """
    toks = ["<start>"]
    tok = f"STYLE_{style_label}"; assert tok in vocab.token_to_id, tok
    toks += ["<style>", tok]
    tok = f"TONALITY_{LT_TONALITY}"; assert tok in vocab.token_to_id, tok
    toks += ["<tonality>", tok]
    tok = f"TYPE_{type_label}"; assert tok in vocab.token_to_id, tok
    toks.append(tok)
    tok = f"FORM_{LT_FORM}"; assert tok in vocab.token_to_id, tok
    toks.append(tok)
    toks.append("|:")

    # First-chord L1 seed
    quality = LT_SEED_QUALITY_BY_TYPE[type_label]
    dur_tok  = f"DUR_{LT_SEED_DUR}"
    root_tok = f"R_{LT_SEED_ROOT}"
    qual_tok = f"Q_{quality}"
    for t in (dur_tok, root_tok, qual_tok):
        assert t in vocab.token_to_id, f"Missing vocab token: {t}"
    toks += [".", dur_tok, root_tok, qual_tok]
    return toks


def _lt_trim_chords(chords: list[dict], max_beats: float) -> list[dict]:
    """Same trim as notebook 13: keep chords starting before `max_beats`;
    clip the final chord so it never rings past the window."""
    kept = []
    for c in chords:
        if c["onset_beats"] >= max_beats:
            break
        end = c["onset_beats"] + c["duration_beats"]
        if end > max_beats:
            c = dict(c, duration_beats=round(max_beats - c["onset_beats"], 4))
        kept.append(c)
    return kept


lt_manifest = []
for idx, (style_tag, style_label, type_label) in enumerate(LT_SELECTION, start=1):
    prompt = _lt_build_prompt(style_label, type_label)

    ids, toks, eig = generate_v3(
        model, vocab, eigen_computer, prompt,
        max_new_tokens=LT_MAX_TOKENS,
        temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
        device=device, eigen_prior=eigen_prior,
    )

    decoded = tokenizer.decode(toks)
    trimmed = _lt_trim_chords(decoded, LT_MAX_BEATS)
    if not trimmed:
        print(f"[{idx:02d}] {style_label:>5s} × type_{type_label:<12s}  ⚠ no chords decoded, skipping")
        continue
    actual_end = trimmed[-1]["onset_beats"] + trimmed[-1]["duration_beats"]

    global_idx = 8 + idx  # model_A clips occupy 09..16
    stem    = f"{global_idx:02d}_{LT_SOURCE}_{idx:02d}_{style_tag}__type_{type_label}"
    midi_out = LT_MIDI_DIR  / f"{stem}.mid"
    wav_out  = LT_AUDIO_DIR / f"{stem}.wav"

    tokenizer.chords_to_midi(trimmed, str(midi_out), tpb=960, tempo_bpm=LT_TEMPO_BPM)

    audio, sr = pm.render_mpe_to_audio_data(
        str(midi_out),
        speed=LT_RENDER_SPEED,
        waveform=LT_WAVEFORM,
        reverb=LT_REVERB,
        save_path=str(wav_out),
    )
    n_samples    = audio.shape[-1] if audio.ndim == 2 else len(audio)
    duration_sec = n_samples / sr

    seed_quality = LT_SEED_QUALITY_BY_TYPE[type_label]
    lt_manifest.append({
        "id": f"{LT_SOURCE}_{idx:02d}",
        "source": LT_SOURCE,
        "style": style_tag,
        "style_token": f"STYLE_{style_label}",
        "transformation": f"type_{type_label}",
        "type_token": f"TYPE_{type_label}",
        "tonality": LT_TONALITY,
        "form": LT_FORM,
        "seed_chord": f". DUR_{LT_SEED_DUR} R_{LT_SEED_ROOT} Q_{seed_quality}",
        "trimmed_midi": str(midi_out.relative_to(ROOT_DIR)),
        "audio": str(wav_out.relative_to(ROOT_DIR)),
        "bars": LT_N_BARS,
        "beats_per_bar": LT_BEATS_PER_BAR,
        "tempo_bpm": LT_TEMPO_BPM,
        "n_chords": len(trimmed),
        "end_beat": round(actual_end, 2),
        "duration_sec": round(duration_sec, 2),
        "sampling": {"temperature": TEMPERATURE, "top_k": TOP_K, "top_p": TOP_P},
        "render": {
            "waveform": LT_WAVEFORM,
            "speed": LT_RENDER_SPEED,
            "reverb_pct": LT_REVERB,
            "sample_rate": sr,
        },
    })

    print(f"[{idx:02d}] {style_label:>5s} × type_{type_label:<12s}  "
          f"seed=Q_{seed_quality:<7s}  chords={len(trimmed):3d}  "
          f"end={actual_end:5.1f}b  → {wav_out.name}")

# Manifest with the same shape as the dataset manifest (notebook 13)
LT_SURVEY_QUESTIONS = [
    {"key": "harmony",      "label": "Harmony",      "prompt": "How coherent do you find the harmonic motion?"},
    {"key": "plausibility", "label": "Plausibility", "prompt": "How plausible is this chord progression for a potential song?"},
    {"key": "dissonance",   "label": "Dissonance",   "prompt": "How dissonant is this chord progression?"},
    {"key": "novelty",      "label": "Novelty",      "prompt": "How surprising or novel do you find this progression?"},
]
for e in lt_manifest:
    e["ratings"] = {q["key"]: None for q in LT_SURVEY_QUESTIONS}

lt_manifest_doc = {
    "source": LT_SOURCE,
    "checkpoint": str(CHECKPOINT_PATH.relative_to(ROOT_DIR)),
    "tuning": "53-EDO (MPE)",
    "n_bars": LT_N_BARS,
    "tempo_bpm": LT_TEMPO_BPM,
    "styles": sorted({e["style"] for e in lt_manifest}),
    "transformations": sorted({e["transformation"] for e in lt_manifest}),
    "seed_quality_by_type": LT_SEED_QUALITY_BY_TYPE,
    "survey_questions": LT_SURVEY_QUESTIONS,
    "clips": lt_manifest,
}

lt_manifest_path = LT_OUT_DIR / "manifest.json"
lt_manifest_path.write_text(json.dumps(lt_manifest_doc, indent=2, ensure_ascii=False))
print(f"\nwrote {lt_manifest_path}  ({len(lt_manifest)} {LT_SOURCE} clips)")
